# Cálculo de la anchura de aceras a partir de polígonos GeoJSON

**Objetivo:** dado un archivo GeoJSON con las **geometrías poligonales** de las aceras, estimar su anchura cada **1 metro** a lo largo de su eje central. No requiere un archivo de portales, calles ni una columna previa de anchura.

**Procedimiento original:** extraído y simplificado de la **celda 31 de `Accesibilidad (2).ipynb`** y de la **celda 3 de `Calles (1).ipynb`** (celdas contadas desde 1).

1. Leer los polígonos de aceras y reproyectarlos a **EPSG:25830**, en metros (adecuado para Madrid).
2. Calcular su eje central con `pygeoops.centerline`.
3. Tomar muestras cada 1 m sobre cada eje central.
4. Trazar una sección perpendicular al eje en cada muestra.
5. Intersectar la sección con el polígono y medir la longitud del tramo que atraviesa la muestra.
6. Exportar puntos con `anchura_m`, identificador de acera, posición y clasificación opcional.

**Qué significa «anchura real»:** es una **estimación geométrica de la anchura representada por el polígono**. No equivale necesariamente a la anchura libre de paso: bolardos, alcorques, terrazas u otros obstáculos no descontados en la capa de entrada **no se detectan**. Las irregularidades y ramificaciones de los polígonos también pueden afectar al cálculo.

**Entrada mínima:** `aceras_madrid.geojson` en la misma carpeta que este notebook, o cambiar `RUTA_ACERAS`. **Salida:** un GeoPackage de puntos (o GeoJSON, si se configura), listo para visualizar en QGIS.

**Dependencias:** `geopandas`, `shapely` (v2+), `numpy`, `pandas`, `pygeoops`, `tqdm`, `matplotlib` (solo para la vista previa). Instalar en el entorno si faltan:

```bash
pip install geopandas shapely numpy pandas pygeoops tqdm matplotlib
```

## 1. Configuración

Solo tienes que modificar `RUTA_ACERAS` si el GeoJSON está en otra carpeta. Para hacer una prueba rápida, pon `MAX_ACERAS = 20`; para procesar todas, usa `None`.

In [ ]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import LineString, Point
from shapely.validation import make_valid
from tqdm.auto import tqdm

try:
    import pygeoops
except ImportError as exc:
    raise ImportError(
        'Falta pygeoops. Ejecuta en una celda: %pip install pygeoops'
    ) from exc

# GeoJSON de polígonos (Polygon / MultiPolygon) de aceras.
RUTA_ACERAS = Path('aceras_madrid.geojson')

# Madrid: ETRS89 / UTM 30N. Todas las distancias se calculan en metros.
CRS_METROS = 'EPSG:25830'

# Resolución y parámetros de la metodología original.
PASO_M = 1.0
SEMILONGITUD_CORTE_M = 20.0   # Perpendicular completa de 40 m
DELTA_TANGENTE_M = 0.5
MIN_ANCHURA_M = 0.3            # Excluir cortes anómalamente pequeños
MAX_ANCHURA_M = 20.0           # Excluir cortes anómalamente grandes

# Opciones: None = todas las aceras; 20 = solo las 20 primeras.
MAX_ACERAS = None

# Outputs. Formato de puntos: .gpkg recomendado para toda la ciudad;
# puede cambiarse a .geojson sin modificar el resto del código.
CARPETA_SALIDA = Path('salidas')
SALIDA_PUNTOS = CARPETA_SALIDA / 'anchura_aceras_puntos_1m.gpkg'
SALIDA_RESUMEN = CARPETA_SALIDA / 'resumen_anchura_por_acera.csv'
EXPORTAR_CORTES = False       # True: guarda también las secciones medidas
EXPORTAR_EJES = False         # True: guarda ejes centrales para comprobación

# Clasificación *orientativa* por anchura geométrica; NO es un
# diagnóstico completo de cumplimiento normativo o accesibilidad.
UMBRAL_1_M80 = 1.80

CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
print('Entrada:', RUTA_ACERAS)
print('Salida:', SALIDA_PUNTOS)

## 2. Funciones geométricas

La geometría se mide mediante una sección perpendicular al eje. Como mejora respecto al código de origen, cuando el corte intersecta **varias zonas desconectadas** del polígono, solo se mide el segmento que pasa por el punto de muestreo; no se suman anchos de tramos separados.

In [ ]:
def extraer_poligonos(geom):
    'Repara geometrías cuando sea posible y devuelve polígonos individuales.'
    if geom is None or geom.is_empty:
        return []
    if not geom.is_valid:
        geom = make_valid(geom)
    if geom is None or geom.is_empty:
        return []
    if geom.geom_type == 'Polygon':
        return [geom] if geom.area > 0 else []
    if geom.geom_type in ('MultiPolygon', 'GeometryCollection'):
        res = []
        for parte in geom.geoms:
            res.extend(extraer_poligonos(parte))
        return res
    return []


def extraer_lineas(geom):
    'Devuelve únicamente partes LineString de una geometría.'
    if geom is None or geom.is_empty:
        return []
    if geom.geom_type == 'LineString':
        return [geom] if geom.length > 0 else []
    if geom.geom_type in ('MultiLineString', 'GeometryCollection'):
        res = []
        for parte in geom.geoms:
            res.extend(extraer_lineas(parte))
        return res
    return []


def crear_ejes_centrales(poligono):
    'Parámetros originales de generación de centerlines.'
    centro = pygeoops.centerline(
        poligono,
        densify_distance=1,
        min_branch_length=2,
        simplifytolerance=0.25,
    )
    return [linea for linea in extraer_lineas(centro) if linea.length >= 1.0]


def corte_perpendicular(eje, distancia_m, semilongitud_m, delta_m):
    'Crea una sección perpendicular a la dirección local del eje.'
    d0 = max(0.0, distancia_m - delta_m)
    d1 = min(eje.length, distancia_m + delta_m)
    if d1 <= d0:
        return None

    p0, p1 = eje.interpolate(d0), eje.interpolate(d1)
    dx, dy = p1.x - p0.x, p1.y - p0.y
    modulo = np.hypot(dx, dy)
    if modulo == 0:
        return None

    ux, uy = -dy / modulo, dx / modulo
    p = eje.interpolate(distancia_m)
    return LineString([
        (p.x - ux * semilongitud_m, p.y - uy * semilongitud_m),
        (p.x + ux * semilongitud_m, p.y + uy * semilongitud_m),
    ])


def segmento_que_pasa_por_punto(interseccion, punto, tolerancia_m=0.01):
    'Evita sumar anchuras de secciones separadas en geometrías complejas.'
    segmentos = extraer_lineas(interseccion)
    candidatos = [s for s in segmentos if s.distance(punto) <= tolerancia_m]
    if not candidatos:
        return None
    # Por robustez ante pequeños errores numéricos, usar la sección más próxima.
    return min(candidatos, key=lambda s: s.distance(punto))

## 3. Leer el GeoJSON de aceras

La entrada debe contener **polígonos**, no puntos ni líneas. GeoJSON suele declararse en EPSG:4326 y aquí se reproyecta a EPSG:25830 para obtener medidas en metros. Si la fuente usa otro CRS, debe estar correctamente declarado: no se asigna arbitrariamente.

In [ ]:
if not RUTA_ACERAS.is_file():
    raise FileNotFoundError(
        f'No existe {RUTA_ACERAS.resolve()}. '
        'Copia el GeoJSON junto al notebook o cambia RUTA_ACERAS.'
    )

aceras = gpd.read_file(RUTA_ACERAS)
if aceras.empty:
    raise ValueError('El GeoJSON no contiene entidades.')
if aceras.crs is None:
    raise ValueError('CRS de entrada desconocido: define correctamente el CRS en el GeoJSON.')

print(f'Entidades de entrada: {len(aceras):,}; CRS: {aceras.crs}')
aceras = aceras.to_crs(CRS_METROS)
if MAX_ACERAS is not None:
    aceras = aceras.head(MAX_ACERAS).copy()

candidatos_id = ['OBJECTID', 'objectid', 'ObjectId', 'acera_id', 'id', 'ID']
COLUMNA_ID = next((c for c in candidatos_id if c in aceras.columns), None)
print('Columna ID:', COLUMNA_ID or '(se usará índice de fila)')

if not any(g is not None and g.geom_type in ('Polygon', 'MultiPolygon', 'GeometryCollection')
           for g in aceras.geometry):
    raise ValueError('Se esperan geometrías Polygon/MultiPolygon, no puntos o líneas.')

## 4. Calcular la anchura cada metro

La medida está asociada a un **punto sobre el eje central**. Los parámetros de filtrado `MIN_ANCHURA_M` y `MAX_ANCHURA_M` reproducen el criterio del original: los cortes descartados **no** son medidas cero ni se interpolan.

In [ ]:
registros = []
registros_cortes = []
registros_ejes = []
contadores = {'poligonos': 0, 'sin_eje': 0, 'sin_corte_valido': 0, 'errores_centerline': 0}

for i, (_, fila) in enumerate(tqdm(aceras.iterrows(), total=len(aceras), desc='Aceras')):
    valor_id = fila[COLUMNA_ID] if COLUMNA_ID else i
    acera_id = str(valor_id) if pd.notna(valor_id) else str(i)
    poligonos = extraer_poligonos(fila.geometry)

    for parte_id, poligono in enumerate(poligonos):
        contadores['poligonos'] += 1
        try:
            ejes = crear_ejes_centrales(poligono)
        except Exception:
            contadores['errores_centerline'] += 1
            continue

        if not ejes:
            contadores['sin_eje'] += 1
            continue

        for eje_id, eje in enumerate(ejes):
            if EXPORTAR_EJES:
                registros_ejes.append({
                    'acera_id': acera_id, 'parte_id': parte_id,
                    'eje_id': eje_id, 'longitud_m': eje.length, 'geometry': eje,
                })

            for distancia_m in np.arange(0.0, eje.length + 1e-9, PASO_M):
                p = eje.interpolate(float(distancia_m))
                corte = corte_perpendicular(
                    eje, float(distancia_m),
                    SEMILONGITUD_CORTE_M, DELTA_TANGENTE_M,
                )
                if corte is None:
                    continue

                interseccion = corte.intersection(poligono)
                segmento = segmento_que_pasa_por_punto(interseccion, p)
                if segmento is None:
                    contadores['sin_corte_valido'] += 1
                    continue

                anchura = float(segmento.length)
                if not (MIN_ANCHURA_M < anchura <= MAX_ANCHURA_M):
                    contadores['sin_corte_valido'] += 1
                    continue

                info = {
                    'acera_id': acera_id,
                    'parte_id': parte_id,
                    'eje_id': eje_id,
                    'dist_m': round(float(distancia_m), 3),
                    'anchura_m': round(anchura, 3),
                    'geometry': p,
                }
                registros.append(info)
                if EXPORTAR_CORTES:
                    registros_cortes.append({**info, 'geometry': segmento})

if not registros:
    raise ValueError(
        'No se pudieron calcular anchuras. Revisa la geometría de entrada, '
        'su CRS y los límites MIN/MAX_ANCHURA_M.'
    )

puntos = gpd.GeoDataFrame(registros, geometry='geometry', crs=CRS_METROS)
puntos['cumple_180'] = puntos['anchura_m'] >= UMBRAL_1_M80
puntos['categoria'] = np.select(
    [puntos['anchura_m'] >= 1.80, puntos['anchura_m'] >= 1.50],
    ['desde_1_80', 'entre_1_50_y_1_80'],
    default='menos_1_50',
)

print(f'Puntos con medida: {len(puntos):,}')
print('Diagnóstico:', contadores)
print(puntos[['anchura_m']].describe().round(2).to_string())

## 5. Exportar resultados

El resultado principal es una capa de **puntos con `anchura_m`**. Además, se genera un CSV agregado por acera y, opcionalmente, GeoJSON con las secciones transversales y los ejes para revisar el método en QGIS.

In [ ]:
def guardar_capa(gdf, ruta, capa=None):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    sufijo = ruta.suffix.lower()
    if sufijo == '.gpkg':
        gdf.to_file(ruta, driver='GPKG', layer=capa or ruta.stem, index=False)
    elif sufijo in ('.geojson', '.json'):
        gdf.to_crs('EPSG:4326').to_file(ruta, driver='GeoJSON', index=False)
    else:
        raise ValueError('Extensión no admitida. Usa .gpkg o .geojson')
    print(f'Guardado: {ruta} ({len(gdf):,} entidades)')


guardar_capa(puntos, SALIDA_PUNTOS, capa='anchura_1m')

resumen = (
    puntos.groupby('acera_id', as_index=False)
    .agg(
        n_puntos=('anchura_m', 'size'),
        anchura_media_m=('anchura_m', 'mean'),
        anchura_min_m=('anchura_m', 'min'),
        anchura_max_m=('anchura_m', 'max'),
        pct_menor_180=('cumple_180', lambda s: round((~s).mean() * 100, 2)),
    )
)
resumen.to_csv(SALIDA_RESUMEN, index=False)
print(f'Guardado: {SALIDA_RESUMEN} ({len(resumen):,} aceras)')

if EXPORTAR_CORTES and registros_cortes:
    cortes = gpd.GeoDataFrame(registros_cortes, geometry='geometry', crs=CRS_METROS)
    guardar_capa(cortes, CARPETA_SALIDA / 'anchura_aceras_cortes_1m.geojson')

if EXPORTAR_EJES and registros_ejes:
    ejes_gdf = gpd.GeoDataFrame(registros_ejes, geometry='geometry', crs=CRS_METROS)
    guardar_capa(ejes_gdf, CARPETA_SALIDA / 'ejes_centrales_aceras.geojson')

print('\nPrimeras filas:')
print(resumen.head(10).round(2).to_string(index=False))

## 6. Vista previa opcional

Muestra las primeras mediciones sobre la geometría original para comprobar que los ejes y cortes tienen sentido. No modifica las salidas.

In [ ]:
import matplotlib.pyplot as plt

id_ejemplo = puntos['acera_id'].iloc[0]
muestra = puntos.loc[puntos['acera_id'] == id_ejemplo].head(100)

if COLUMNA_ID:
    base = aceras.loc[aceras[COLUMNA_ID].astype(str) == id_ejemplo]
else:
    base = aceras.iloc[[int(id_ejemplo)]]

fig, ax = plt.subplots(figsize=(10, 5))
base.plot(ax=ax, color='#d8e7ed', edgecolor='#445566', linewidth=1)
muestra.plot(ax=ax, column='anchura_m', cmap='viridis', legend=True,
            markersize=18, zorder=3, legend_kwds={'label': 'Anchura estimada (m)'})
ax.set_aspect('equal')
ax.set_title(f'Anchura geométrica estimada – acera {id_ejemplo}')
ax.set_axis_off()
plt.show()

### Limitaciones y recomendaciones para interpretar resultados

- **Anchura geométrica ≠ anchura libre de obstáculos.** El modelo solo detecta estrechamientos representados en el GeoJSON poligonal.
- Las secciones pueden ser inestables en cruces, esquinas, ensanchamientos y ramificaciones; conviene revisar visualmente una muestra con `EXPORTAR_CORTES = True`.
- Se usa EPSG:25830 por tratarse de Madrid. Para otra ciudad, elige su **CRS proyectado en metros**.
- `pygeoops.centerline` genera una aproximación del eje; no siempre coincide con el recorrido peatonal real.
- El umbral de 1,80 m es una **etiqueta de anchura**; por sí solo no determina el cumplimiento de todas las condiciones legales de accesibilidad.
- Para procesar toda Madrid, se recomienda guardar el resultado en **GPKG**; GeoJSON puede ocupar bastante espacio.